In [ ]:
from datetime import datetime
import math


def _parse_amount(value, error_message):
    """Привести значение к float или бросить ValueError.

    Гарантирует, что для невалидных входных данных
    (строка, None, inf, nan и т.п.) вылетит именно ValueError.
    """
    if isinstance(value, bool):
        raise ValueError(error_message)
    try:
        result = float(value)
    except (TypeError, ValueError):
        raise ValueError(error_message) from None

    if not math.isfinite(result):
        raise ValueError(error_message)

    return result


class Account:
    """Банковский счёт с журналом всех операций."""

    def __init__(self, account_holder: str, balance=0):
        message = (
            "Начальный баланс должен быть конечным числом "
            "не меньше 0."
        )
        balance = _parse_amount(balance, message)
        if balance < 0:
            raise ValueError(message)
 
        self.holder = account_holder
        self._balance = balance
        self.operations_history = []

    def _record_operation(self, operation_type, amount, status):
        operation = {
            "type": operation_type,
            "amount": amount,
            "timestamp": datetime.now(),
            "balance": self._balance,
            "status": status,
        }
        self.operations_history.append(operation)
        return operation

    def deposit(self, amount):
        message = (
            "Сумма пополнения должна быть конечным "
            "положительным числом."
        )
        amount = _parse_amount(amount, message)
        if amount <= 0:
            raise ValueError(message)

        self._balance += amount
        self._record_operation("deposit", amount, "success")
        return True

    def withdraw(self, amount):
        message = (
            "Сумма снятия должна быть конечным "
            "положительным числом."
        )
        amount = _parse_amount(amount, message)
        if amount <= 0:
            raise ValueError(message)

        if amount > self._balance:
            self._record_operation("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._record_operation("withdraw", amount, "success")
        return True

    def get_balance(self):
        return self._balance

    def get_history(self):
        return [
            operation.copy() for operation in self.operations_history
        ]

In [27]:
class CreditAccount(Account):
    """Кредитный счёт: баланс может уходить в минус до лимита."""

    def __init__(self, account_holder, balance=0, credit_limit=0):
        limit_message = (
            "Кредитный лимит должен быть конечным числом "
            "не меньше 0."
        )
        credit_limit = _parse_amount(credit_limit, limit_message)
        if credit_limit < 0:
            raise ValueError(limit_message)

        balance_message = (
            "Баланс должен быть конечным числом не ниже "
            "-credit_limit."
        )
        balance = _parse_amount(balance, balance_message)
        if balance < -credit_limit:
            raise ValueError(balance_message)

        super().__init__(account_holder)
        self._balance = balance
        self.credit_limit = credit_limit

    def withdraw(self, amount):
        message = (
            "Сумма снятия должна быть конечным "
            "положительным числом."
        )
        amount = _parse_amount(amount, message)
        if amount <= 0:
            raise ValueError(message)

        if amount > self._balance + self.credit_limit:
            self._record_operation("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._record_operation("withdraw", amount, "success")
        return True

    def get_available_funds(self):
        return self._balance + self.credit_limit

    def _record_operation(self, operation_type, amount, status):
        operation = super()._record_operation(
            operation_type, amount, status
        )
        operation["credit_used"] = (
            operation_type == "withdraw"
            and status == "success"
            and operation["balance"] < 0
        )
        return operation

In [ ]:
credit = CreditAccount("Иван", balance=1000, credit_limit=5000)

print("Кредитный счёт\n")

print(f"Снятие 1500: {credit.withdraw(1500)}")
print(f"Баланс: {credit.get_balance()}")
print(f"Доступно для снятия: {credit.get_available_funds()}\n")

print(f"Снятие 5000: {credit.withdraw(5000)}")
print(f"Баланс после отказа: {credit.get_balance()}\n")

print(f"Пополнение 500: {credit.deposit(500)}")
print(f"Баланс после пополнения: {credit.get_balance()}")
print(f"Доступно для снятия: {credit.get_available_funds()}\n")

print("История операций:")
for i, op in enumerate(credit.get_history(), 1):
    dt = op["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"{i}. {dt} | "
        f"{op['type']:<8} | "
        f"{op['amount']!s:>10} | "
        f"{op['status']:<7} | "
        f"баланс: {op['balance']!s:>8} | "
        f"credit_used: {op['credit_used']}"
    )



Кредитный счёт

Снятие 1500: True
Баланс: -500.0
Доступно для снятия: 4500.0

Снятие 5000: False
Баланс после отказа: -500.0

Пополнение 500: True
Баланс после пополнения: 0.0
Доступно для снятия: 5000.0

История операций:
1. 2026-11-08 13:46:45 | withdraw |     1500.0 | success | баланс:   -500.0 | credit_used: True
2. 2026-11-08 13:46:45 | withdraw |     5000.0 | fail    | баланс:   -500.0 | credit_used: False
3. 2026-11-08 13:46:45 | deposit  |      500.0 | success | баланс:      0.0 | credit_used: False
